# 01 - Extração Distribuída no Cluster Spark (ADLS Gen2)
**Squad 2 — Real Time for Business | Dupla 5 (Grupo 5)**  
**Integrantes:** Zaiden Emiliano Segundo Seleme *(assumindo o escopo do Grupo 5)*  
**Tabelas de Escopo:** `ecommerce_rastreamento_entregas` e `ecommerce_enderecos`  
**Branch:** `feat/squad2-dupla5-rastreamento-enderecos`  

### Objetivo:
1. Realizar a leitura distribuída e paralela de todos os micro-lotes `.parquet` depositados no container `raw/real-time-data/`.
2. Extrair os metadados nativos de arquivo (`_metadata.file_path`) para auditoria de origem.
3. Avaliar a volumetria, distribuição de schemas e consistência das chaves primárias (`id_rastreamento` e `id_endereco`).
4. Mapear o catálogo de status logísticos e a dispersão geográfica dos endereços.

In [ ]:
# Instalação de dependências no escopo da sessão Serverless
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity --quiet


In [ ]:
dbutils.library.restartPython()


In [ ]:
# Carregamento seguro e resiliente das variáveis de ambiente
import os
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed",
        "/Workspace/Repos/estagio_ed"
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

assert client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"
print("Credenciais do Service Principal carregadas com sucesso!")


## 1. Dicionário de Opções OAuth para o Conector Spark ABFS

In [ ]:
# Configurações OAuth com FQDN do Service Principal injetadas a cada chamada do Spark
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

caminho_base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data"
caminho_rastreamento = f"{caminho_base_raw}/*/*/*/*/ecommerce_rastreamento.parquet"
caminho_enderecos = f"{caminho_base_raw}/*/*/*/*/ecommerce_enderecos.parquet"

print("Caminhos de origem configurados:")
print(f"  Rastreamento: {caminho_rastreamento}")
print(f"  Endereços:    {caminho_enderecos}")


## 2. Leitura Paralelizada e Distribuída de `ecommerce_rastreamento_entregas`

In [ ]:
from pyspark.sql.functions import col, countDistinct, count, current_timestamp

# Leitura distribuída de todas as janelas temporais de rastreamento com metadados de origem
df_raw_rastreamento = (
    spark.read
    .options(**adls_options)
    .parquet(caminho_rastreamento)
    .withColumn("arquivo_origem", col("_metadata.file_path"))
)

total_rastreamento = df_raw_rastreamento.count()
arquivos_rastreamento = df_raw_rastreamento.select(countDistinct("arquivo_origem")).first()[0]

print(f"=== Extração de Rastreamento de Entregas ===")
print(f"Total de registros lidos: {total_rastreamento}")
print(f"Total de arquivos parquet particionados: {arquivos_rastreamento}")

print("\nSchema inferido de ecommerce_rastreamento_entregas:")
df_raw_rastreamento.printSchema()

display(df_raw_rastreamento.limit(5))


## 3. Leitura Paralelizada e Distribuída de `ecommerce_enderecos`

In [ ]:
# Leitura distribuída de todas as janelas temporais de endereços com metadados de origem
df_raw_enderecos = (
    spark.read
    .options(**adls_options)
    .parquet(caminho_enderecos)
    .withColumn("arquivo_origem", col("_metadata.file_path"))
)

total_enderecos = df_raw_enderecos.count()
arquivos_enderecos = df_raw_enderecos.select(countDistinct("arquivo_origem")).first()[0]

print(f"=== Extração de Endereços ===")
print(f"Total de registros lidos: {total_enderecos}")
print(f"Total de arquivos parquet particionados: {arquivos_enderecos}")

print("\nSchema inferido de ecommerce_enderecos:")
df_raw_enderecos.printSchema()

display(df_raw_enderecos.limit(5))


## 4. Análise Exploratória e Validação Inicial de Qualidade

In [ ]:
# 1. Unicidade das chaves primárias no RAW
rastreio_unicos = df_raw_rastreamento.select(countDistinct("id_rastreamento")).first()[0]
enderecos_unicos = df_raw_enderecos.select(countDistinct("id_endereco")).first()[0]

print(f"Unicidade de Chaves:")
print(f"  id_rastreamento únicos: {rastreio_unicos} / {total_rastreamento} registros")
print(f"  id_endereco únicos:     {enderecos_unicos} / {total_enderecos} registros")

# 2. Distribuição dos Status Logísticos de Entrega
print("\nDistribuição de Status de Entrega:")
df_raw_rastreamento.groupBy("status_entrega").agg(count("*").alias("total")).orderBy(col("total").desc()).show(truncate=False)

# 3. Distribuição Geográfica de Endereços por Estado (Top 10)
print("\nDistribuição Geográfica dos Endereços por Estado:")
df_raw_enderecos.groupBy("estado").agg(count("*").alias("total")).orderBy(col("total").desc()).show(10, truncate=False)

print("Extração distribuída e auditoria exploratória concluídas com sucesso!")
